In [2]:
import os, shutil, zipfile
from pathlib import Path

PROJECT = Path.home() / "Desktop" / "Project" / "sales-forecasting-assistant"
DOWNLOADS = Path.home() / "Downloads"
RAW_DIR = PROJECT / "data" / "raw"

os.chdir(PROJECT)                      # work from the project folder from now on
RAW_DIR.mkdir(parents=True, exist_ok=True)
(PROJECT / "src" / "ingest").mkdir(parents=True, exist_ok=True)

zip_file = DOWNLOADS / "store-sales-time-series-forecasting.zip"
unzipped = DOWNLOADS / "store-sales-time-series-forecasting"

if zip_file.exists():
    zipfile.ZipFile(zip_file).extractall(RAW_DIR)
elif unzipped.exists():                # Mac sometimes unzips automatically
    for f in unzipped.glob("*.csv"):
        shutil.copy(f, RAW_DIR)
else:
    print("Could not find the Kaggle download in Downloads - check the file name")

print("Working in:", os.getcwd())
print("Files in data/raw:", sorted(p.name for p in RAW_DIR.glob("*.csv")))


Working in: /Users/dayoawe1/Desktop/Project/sales-forecasting-assistant
Files in data/raw: ['holidays_events.csv', 'oil.csv', 'sample_submission.csv', 'stores.csv', 'test.csv', 'train.csv', 'transactions.csv']


In [3]:
%pip install "snowflake-connector-python[pandas]" python-dotenv cryptography

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 3.5 MB/s eta 0:00:00a 0:00:010m
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 5.9 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 2.5 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.6/69.6 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.0/105.0 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 4.5 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.0/137.0 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 606.4 kB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.0/36.0 MB 5.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.2/184.2 kB 6.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 3.4 MB

In [3]:
from cryptography.hazmat.primitives.asymmetric import rsa
from cryptography.hazmat.primitives import serialization

key_dir = Path.home() / ".snowflake"
key_dir.mkdir(exist_ok=True)
key_path = key_dir / "rsa_key.p8"

if not key_path.exists():
    key = rsa.generate_private_key(public_exponent=65537, key_size=2048)
    key_path.write_bytes(key.private_bytes(
        serialization.Encoding.PEM,
        serialization.PrivateFormat.PKCS8,
        serialization.NoEncryption()))

private_key = serialization.load_pem_private_key(key_path.read_bytes(), password=None)
public_pem = private_key.public_key().public_bytes(
    serialization.Encoding.PEM,
    serialization.PublicFormat.SubjectPublicKeyInfo).decode()

public_key_one_line = "".join(l for l in public_pem.splitlines() if "-----" not in l)
print("Copy everything on the next line:\n")
print(public_key_one_line)

Copy everything on the next line:

MIIBIjANBgkqhkiG9w0BAQEFAAOCAQ8AMIIBCgKCAQEAsmAVB6oZWlOOUD0xt+T1/mzrsFIe4VSpmpDqG+v19cGPYNM4/xN6WQaQWrewTsuwfQ1LDXOybWDd1VNUlIw9M8tKoVXwvf6wnG4nYbXhAYYI28MPFiR7wLljlV9008TpHR9ikMZd+jr8p7qRQHM1Isl7VZnC4eAjtY7j1/DKMK+xqF8A0+I2YzY1xFTmnB6zx6NJQPdgWaoyFJmJK7gXUJ38cUOcfGtAk3Tdct2coi//6qPeXg62cNn4fvOMhZV+tbjI6TcfbwSfAOH8ZATEYT5lupxNj4HlR1MZ8JSTzF8WLF/a+pPddBKurJhqf1j6yYXYTrAuN8VCLtJUFEWIwwIDAQAB


In [4]:
ACCOUNT = "GIJIJNG-DTC96224"   # <-- your account identifier

(PROJECT / ".env").write_text(
    f"SNOWFLAKE_ACCOUNT={ACCOUNT}\n"
    "SNOWFLAKE_USER=DAYOAWE1\n"
    "SNOWFLAKE_PRIVATE_KEY_FILE=~/.snowflake/rsa_key.p8\n"
)
print((PROJECT / ".env").read_text())

SNOWFLAKE_ACCOUNT=GIJIJNG-DTC96224
SNOWFLAKE_USER=DAYOAWE1
SNOWFLAKE_PRIVATE_KEY_FILE=~/.snowflake/rsa_key.p8



In [5]:
%%writefile src/ingest/load_raw.py
"""Load the Kaggle Store Sales CSVs into Snowflake SALES_DB.RAW."""
import os
import pandas as pd
import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas
from dotenv import load_dotenv

load_dotenv()

DATA_DIR = "data/raw"
FILES = {
    "train.csv": "TRAIN",
    "test.csv": "TEST",
    "stores.csv": "STORES",
    "oil.csv": "OIL",
    "holidays_events.csv": "HOLIDAYS_EVENTS",
    "transactions.csv": "TRANSACTIONS",
}


def get_connection():
    return snowflake.connector.connect(
        account=os.environ["SNOWFLAKE_ACCOUNT"],
        user=os.environ["SNOWFLAKE_USER"],
        private_key_file=os.path.expanduser(os.environ["SNOWFLAKE_PRIVATE_KEY_FILE"]),
        role="SYSADMIN",
        warehouse="SALES_WH",
        database="SALES_DB",
        schema="RAW",
    )


def main():
    conn = get_connection()
    try:
        for filename, table in FILES.items():
            df = pd.read_csv(os.path.join(DATA_DIR, filename), dtype=str)
            df.columns = [c.upper() for c in df.columns]
            success, _, nrows, _ = write_pandas(
                conn, df, table_name=table,
                auto_create_table=True, overwrite=True, quote_identifiers=False,
            )
            print(f"{table:<16} {'OK' if success else 'FAILED'}  {nrows:>10,} rows")
    finally:
        conn.close()


if __name__ == "__main__":
    main()

Writing src/ingest/load_raw.py


In [6]:
%run src/ingest/load_raw.py

TRAIN            OK   3,000,888 rows
TEST             OK      28,512 rows
STORES           OK          54 rows
OIL              OK       1,218 rows
HOLIDAYS_EVENTS  OK         350 rows
TRANSACTIONS     OK      83,488 rows
